# Phase 5 

### chargement de la dataset 

In [1]:
import os
import json
import joblib
import numpy as np
import pandas as pd
from flask import Flask, request, jsonify
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")
print("Classes :", data.target_names)

print("\nRépartition de la cible :")
print(y.value_counts(normalize=True).sort_index().round(3))

Classes : ['malignant' 'benign']

Répartition de la cible :
target
0    0.373
1    0.627
Name: proportion, dtype: float64


### Séparation des données : Train / test split

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

print("Train :", len(X_train))
print("Test :", len(X_test))

Train : 455
Test : 114


### Normalisation et entraînement

#### Entraîner le scaler et le modèle

In [4]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

modele = LogisticRegression(max_iter=5000, random_state=42)
modele.fit(X_train_scaled, y_train)

y_pred = modele.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy sur le test : {accuracy:.3f}")

Accuracy sur le test : 0.982


### fonction sauvegarder_modele

In [10]:
def sauvegarder_modele(modele, scaler, chemin="models/modele.joblib"):
    

    os.makedirs(os.path.dirname(chemin), exist_ok=True)

    joblib.dump(
        {
            "modele": modele,
            "scaler": scaler,
            "features_names": list(X.columns),
            "target_names": list(data.target_names)
        },
        chemin
    )

sauvegarder_modele(modele=modele,scaler=scaler,chemin="models/modele.joblib")

#### Recharger le fichier .joblib

In [12]:
objets_charges = joblib.load("models/modele.joblib")

modele_charge = objets_charges["modele"]
scaler_charge = objets_charges["scaler"]
features_names = objets_charges["features_names"]
target_names = objets_charges["target_names"]
print("Nombre de features attendues :", len(features_names))


Nombre de features attendues : 30


### Création de l’API Flask et endpoint /predict

In [20]:
app = Flask(__name__)

objets_api = joblib.load("models/modele.joblib")

modele_api = objets_api["modele"]
scaler_api = objets_api["scaler"]
features_names_api = objets_api["features_names"]
target_names_api = objets_api["target_names"]

@app.route("/predict", methods=["POST"])
def predict():
    """
    Reçoit un JSON {"features": [...]}
    Renvoie une prédiction, une probabilité et un label.
    """

    data_json = request.get_json()

    if data_json is None:
        return jsonify({"erreur": "La requête doit contenir un JSON."}), 400

    if "features" not in data_json:
        return jsonify({"erreur": "La clé 'features' est obligatoire."}), 400

    features = data_json["features"]

    if not isinstance(features, list):
        return jsonify({"erreur": "'features' doit être une liste."}), 400

    if len(features) == 0:
        return jsonify({"erreur": "La liste 'features' ne doit pas être vide."}), 400

    if len(features) != len(features_names_api):
        return jsonify({
            "erreur": f"Nombre de valeurs incorrect. Attendu : {len(features_names_api)}."
        }), 400

    try:
        features_array = np.array(features, dtype=float).reshape(1, -1)
    except ValueError:
        return jsonify({"erreur": "Toutes les valeurs doivent être numériques."}), 400

    features_df = pd.DataFrame(features_array, columns=features_names_api)
    features_scaled = scaler_api.transform(features_df)

    prediction = int(modele_api.predict(features_scaled)[0])
    proba = float(modele_api.predict_proba(features_scaled)[0][prediction])
    label = target_names_api[prediction]

    return jsonify({
        "prediction": prediction,
        "proba": round(proba, 3),
        "label": label
    })

#### Test normal de l’API

In [24]:
client = app.test_client()

features_exemple = X_test.iloc[0].tolist()

reponse_normale = client.post("/predict",json={"features": features_exemple})

print("Code HTTP :", reponse_normale.status_code)
print("Données brutes :")
print(reponse_normale.data.decode("utf-8"))

Code HTTP : 200
Données brutes :
{"label":"malignant","prediction":0,"proba":1.0}



#### Test limite sans clé features

In [23]:
reponse_sans_features = client.post("/predict",json={"valeurs": features_exemple})

print("Code HTTP :", reponse_sans_features.status_code)
print("Réponse JSON :")
print(reponse_sans_features.get_json())

Code HTTP : 400
Réponse JSON :
{'erreur': "La clé 'features' est obligatoire."}


#### Test limite avec mauvais nombre de valeurs

In [25]:
reponse_mauvais_nombre = client.post("/predict",json={"features": [1, 2, 3]})

print("Code HTTP :", reponse_mauvais_nombre.status_code)
print("Réponse JSON :")
print(reponse_mauvais_nombre.get_json())

Code HTTP : 400
Réponse JSON :
{'erreur': 'Nombre de valeurs incorrect. Attendu : 30.'}


#### Test adversarial avec texte

In [26]:
features_texte = ["texte"] * len(features_names_api)

reponse_texte = client.post("/predict",json={"features": features_texte})

print("Code HTTP :", reponse_texte.status_code)
print("Réponse JSON :")
print(reponse_texte.get_json())

Code HTTP : 400
Réponse JSON :
{'erreur': 'Toutes les valeurs doivent être numériques.'}


#### Test adversarial avec tableau vide

In [27]:
reponse_vide = client.post("/predict",json={"features": []})

print("Code HTTP :", reponse_vide.status_code)
print("Réponse JSON :")
print(reponse_vide.get_json())

Code HTTP : 400
Réponse JSON :
{'erreur': "La liste 'features' ne doit pas être vide."}


### Création d’un fichier API

#### Écrire le fichier api/app.py

In [29]:
os.makedirs("api", exist_ok=True)

code_api = '''
import joblib
import numpy as np

from flask import Flask, request, jsonify

app = Flask(__name__)

objets_api = joblib.load("models/modele.joblib")

modele_api = objets_api["modele"]
scaler_api = objets_api["scaler"]
features_names_api = objets_api["features_names"]
target_names_api = objets_api["target_names"]


@app.route("/predict", methods=["POST"])
def predict():
    data_json = request.get_json()

    if data_json is None:
        return jsonify({"erreur": "La requête doit contenir un JSON."}), 400

    if "features" not in data_json:
        return jsonify({"erreur": "La clé 'features' est obligatoire."}), 400

    features = data_json["features"]

    if not isinstance(features, list):
        return jsonify({"erreur": "'features' doit être une liste."}), 400

    if len(features) == 0:
        return jsonify({"erreur": "La liste 'features' ne doit pas être vide."}), 400

    if len(features) != len(features_names_api):
        return jsonify({
            "erreur": f"Nombre de valeurs incorrect. Attendu : {len(features_names_api)}."
        }), 400

    try:
        features_array = np.array(features, dtype=float).reshape(1, -1)
    except ValueError:
        return jsonify({"erreur": "Toutes les valeurs doivent être numériques."}), 400

    features_scaled = scaler_api.transform(features_array)

    prediction = int(modele_api.predict(features_scaled)[0])
    proba = float(modele_api.predict_proba(features_scaled)[0][prediction])
    label = target_names_api[prediction]

    return jsonify({
        "prediction": prediction,
        "proba": round(proba, 3),
        "label": label
    })


if __name__ == "__main__":
    app.run(debug=True)
'''

with open("api/app.py", "w", encoding="utf-8") as fichier:
    fichier.write(code_api)



#### test avec curl

In [31]:
exemple_json = {"features": features_exemple}


print(json.dumps(exemple_json, indent=2))

{
  "features": [
    19.55,
    28.77,
    133.6,
    1207.0,
    0.0926,
    0.2063,
    0.1784,
    0.1144,
    0.1893,
    0.06232,
    0.8426,
    1.199,
    7.158,
    106.4,
    0.006356,
    0.04765,
    0.03863,
    0.01519,
    0.01936,
    0.005252,
    25.05,
    36.27,
    178.6,
    1926.0,
    0.1281,
    0.5329,
    0.4251,
    0.1941,
    0.2818,
    0.1005
  ]
}
